In [2]:
"""
What is Async and Why Use It?
Asynchronous programming allows multiple operations to run concurrently without blocking each other. When making API calls:
------------------------------------------
Synchronous: Each request must complete before the next one starts
Asynchronous: Multiple requests can be "in flight" simultaneously

------------------------------------------
This is especially useful when:
Making many API calls in parallel
Handling long-running operations without blocking
Building responsive applications

-------------------------------------------
async/await: Python keywords for writing asynchronous code
AsyncOpenAI: The async version of the OpenAI client
asyncio.gather(): For running multiple async operations in parallel
"""

'\nWhat is Async and Why Use It?\nAsynchronous programming allows multiple operations to run concurrently without blocking each other. When making API calls:\n------------------------------------------\nSynchronous: Each request must complete before the next one starts\nAsynchronous: Multiple requests can be "in flight" simultaneously\n\n------------------------------------------\nThis is especially useful when:\nMaking many API calls in parallel\nHandling long-running operations without blocking\nBuilding responsive applications\n\n-------------------------------------------\nasync/await: Python keywords for writing asynchronous code\nAsyncOpenAI: The async version of the OpenAI client\nasyncio.gather(): For running multiple async operations in parallel\n'

In [12]:
# Sending a prompt to an Ollama model and asking it to answer as a fictional/persona character in JSON format.
import os
from getpass import getpass
import time
import json
import asyncio
import os, sys
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")

# "Pretend you are Sophia, use her background to shape your response, 
# speak in first person, give a short personal-style reaction, and return the result as JSON.

from llm_config import ollama, MODEL_OLLAMA, API_KEY,OLLAMA_BASE_URL
async_ollama = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=API_KEY
)

system_prompt = """
You are answering as a fictional character.

Return ONLY a valid JSON object.
Do not include markdown.
Do not include any text before or after the JSON.

The JSON must contain exactly two fields:
- "reason": a short explanation of the character's stated preference
- "choice": either "A" or "B"
"""

user_query = """
Choose between:

A) Candidate A
B) Candidate B

Reply only in JSON.
"""
start_time = time.time()

response = ollama.responses.create(
    model=MODEL_OLLAMA,
    instructions=system_prompt,
    input=user_query,
    text={"format": {"type": "json_object"}}          # "Whatever answer you generate, return it as a JSON object."
)

print(response.output_text)
end_time = time.time()
print(f"Time taken: {end_time - start_time:.2f} seconds")

{
  "reason": "I am more comfortable with complex rulesets",
  "choice": "B"
Time taken: 5.05 seconds


In [ ]:
# Synchronus run -- sequential execution -- This function simply runs the same AI question 5 times, measures how long each run takes, and 
# counts how many times the model chooses A or B.
"""
Query 1   █████ 5 sec
Query 2        █████ 5 sec
Query 3             █████ 5 sec
Query 4                  █████ 5 sec
...
Query 10                              █████ 5 sec

Total ≈ 50 seconds
"""

def run_several_times(runs=5):
    total_time = 0
    votes = {"A": 0, "B": 0} # Track votes for Gavin Newsom (A) and J.D. Vance (B)
    
    for i in range(runs):
        start_time = time.time()
        response = ollama.responses.create(
            model=MODEL_OLLAMA,
            instructions=system_prompt,
            input=user_query,
            text={"format": {"type": "json_object"}}
        )
        
        end_time = time.time()
        time_taken = end_time - start_time
        total_time += time_taken
        
        print(f"\n--- Run {i + 1} ---")
        print("RAW OUTPUT:")
        print(repr(response.output_text))

        # Check for empty response
        if not response.output_text:
            print("EMPTY OUTPUT - skipping this run")
            continue
        # Parse response and count vote
        # Convert JSON text to Python dictionary
        try:
            response_json = json.loads(response.output_text)

        except json.JSONDecodeError as e:
            print("INVALID JSON - skipping this run")
            print("Error:", e)
            continue
        vote = str(response_json.get('vote') or '').strip().upper()
        # Accept "A", "B", or strings that start with the letter (e.g. "A) Gavin Newsom")
        vote = vote[:1] if vote[:1] in votes else vote
        if vote in votes:
            votes[vote] += 1
    
    avg_time = total_time / runs
    print(f"\nResults after {runs} runs:")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per run: {avg_time:.2f} seconds")
    print(f"\nVote Tally:")
    print(f"Gavin Newsom (A): {votes['A']} votes")
    print(f"J.D. Vance (B): {votes['B']} votes")

# Run the function
run_several_times()       


--- Run 1 ---
RAW OUTPUT:
'{"reason": "I\'m not really into this game, I need a break from all these choices", "choice": "B"}'

--- Run 2 ---
RAW OUTPUT:
'{"reason":"To make things interesting", "choice":"A"}'

--- Run 3 ---
RAW OUTPUT:
'{"reason": "Candidate B", "choice": "B"}'

--- Run 4 ---
RAW OUTPUT:
'{"reason": "The world needs fresh ideas and diverse perspectives", "choice": "A"}'

--- Run 5 ---
RAW OUTPUT:
'{"reason":"I\'m not into politics", "choice":"A"}'

Results after 5 runs:
Total time: 8.91 seconds
Average time per run: 1.78 seconds

Vote Tally:
Gavin Newsom (A): 0 votes
J.D. Vance (B): 0 votes


In [ ]:
| Step | Code / keyword           | Simple meaning                                                                |
| ---- | ------------------------ | ----------------------------------------------------------------------------- |
| 1    | `async def`              | Define a function that can run asynchronously                                 |
| 2    | `await`                  | Wait for an async operation to finish, while allowing other async work to run |
| 3    | `async_client`           | The asynchronous API client                                                   |
| 4    | `tasks = [...]`          | Create multiple async operations                                              |
| 5    | `for _ in range(n)`      | Repeat the operation `n` times                                                |
| 6    | `*tasks`                 | Unpack the list into individual tasks                                         |
| 7    | `asyncio.gather()`       | Run/wait for multiple async operations together                               |
| 8    | `await asyncio.gather()` | Wait until all operations finish                                              |
| 9    | `results`                | Store all returned results                                                    |
| 10   | `asyncio.run()`          | Start an async program from a normal Python `.py` file                        |


In [17]:
# Asynchonous run - run all 5 cocurrently or in parallel and finish it all sooner 
"""
make_single_query()
        ↓
ONE API request
        ↓
returns (vote, time)
----
run_multiple_queries_async()
        ↓
creates 5 make_single_query()
        ↓
asyncio.gather()
        ↓
runs them concurrently
        ↓
gets 5 results

Query 1   █████
Query 2   █████
Query 3   █████
Query 4   █████
Query 5   █████
Total ≈ 5–6 seconds
"""
from llm_config import ollama, MODEL_OLLAMA, API_KEY,OLLAMA_BASE_URL
async_ollama = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=API_KEY
)
async def make_single_query():                    # This defines one asynchronous query.
    start_time = time.time()
    response = await async_ollama.responses.create(      # asynchronous client.
        model=MODEL_OLLAMA,
        instructions=system_prompt,
        input=user_query,
        text={"format": {"type": "json_object"}}
    )
    end_time = time.time()
    time_taken = end_time - start_time
    # Parse response and get vote
    response_json = json.loads(response.output_text)
    vote = str(response_json.get('vote') or '').strip().upper()
    vote = vote[:1] if vote[:1] in {"A", "B"} else vote
    return vote, time_taken

async def run_multiple_queries_async(num_runs=5):
    start_time = time.time()
   
    tasks = [make_single_query() for _ in range(num_runs)]         # Create list of tasks
    """ above line is a short way of writing below -- if num_runs = 3 then
        tasks = [
        make_single_query(),
        make_single_query(),
        make_single_query()
        ]
    """
    results = await asyncio.gather(*tasks)                         # unpack the list. tasks = [task1, task2, task3...........]
    
    end_time = time.time()
    total_time = end_time - start_time
    votes = {"A": 0, "B": 0}                                       # Track votes for Gavin Newsom (A) and J.D. Vance (B)
    individual_times = []
    
    for vote, time_taken in results:
        if vote in votes:
            votes[vote] += 1
        individual_times.append(time_taken)
    
    avg_individual_time = sum(individual_times) / len(individual_times)
    print(f"\nResults after {num_runs} runs:")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per run: {avg_individual_time:.2f} seconds")
    print(f"\nVote Tally:")
    print(f"Gavin Newsom (A): {votes['A']} votes")
    print(f"J.D. Vance (B): {votes['B']} votes")
await run_multiple_queries_async()             # Jupyter supports top-level await — no asyncio.run() needed


Results after 5 runs:
Total time: 9.67 seconds
Average time per run: 8.01 seconds

Vote Tally:
Gavin Newsom (A): 0 votes
J.D. Vance (B): 0 votes
